# Supplementary Multidimensional Similarity Analysis (Colab)

This revised notebook addresses the issue where ad-level values appeared identical by:

1. **Diagnosing repeated respondent rows** (same person repeated across ad conditions).
2. Building **ad personality vectors** from ad-signal columns (`OPE_cond`, `CON_cond`, `EXT_cond`, `AGR_cond`, `NEU_cond`) when available.
3. Falling back to perceived-ad items only if needed.
4. Using **participant trait vectors** (preferably z-scored) for similarity.

Outputs:
- Table 7: Ad personality profile matrix (5D)
- Table 8: Euclidean vs cosine comparison
- Table 9: Cross-group validation
- Regression and appendix figures


In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Upload data file


In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Loaded:", filename)

# Support CSV, TSV, XLS/XLSX
if filename.lower().endswith((".xls", ".xlsx")):
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
else:
    try:
        df = pd.read_csv(io.BytesIO(uploaded[filename]), low_memory=False)
    except Exception:
        df = pd.read_csv(io.BytesIO(uploaded[filename]), sep="	", low_memory=False)

print("Shape:", df.shape)
df.head(3)


## 2) Clean nulls and set key columns


In [ ]:
null_tokens = ["#NULL!", "", " ", "NA", "N/A", "nan", "NaN", "None"]
df = df.replace(null_tokens, np.nan)

# Participant Big Five (raw + z if available)
trait_raw = ["OPEN_Score", "CONSC_Score", "EXTRA_Score", "AGREE_Score", "NEURO_Score"]
trait_z = ["ZOPEN_Score", "ZCONSC_Score", "ZEXTRA_Score", "ZAGREE_Score", "ZNEURO_Score"]

# Ad signal columns (preferred for Table 7 ad vectors)
ad_signal_cols = ["OPE_cond", "CON_cond", "EXT_cond", "AGR_cond", "NEU_cond"]

click_items = ["CLIC1", "CLIC2", "CLIC3", "CLIC4"]
pers_items = ["PERS1", "PERS2", "PERS3", "PERS4", "PERS5", "PERS6"]
rele_items = ["RELE1", "RELE2", "RELE3"]

for c in set(trait_raw + trait_z + ad_signal_cols + click_items + pers_items + rele_items):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if set(click_items).intersection(df.columns):
    df["CLICK_MEAN"] = df[[c for c in click_items if c in df.columns]].mean(axis=1)
if set(pers_items).intersection(df.columns):
    df["PERSUASIVE_MEAN"] = df[[c for c in pers_items if c in df.columns]].mean(axis=1)
if set(rele_items).intersection(df.columns):
    df["RELEVANCE_MEAN"] = df[[c for c in rele_items if c in df.columns]].mean(axis=1)

print("Columns loaded. AdCond present:", "AdCond" in df.columns)


## 3) Diagnostics: why values looked identical


In [ ]:
id_col = "ResponseId" if "ResponseId" in df.columns else None

participant_trait_cols = trait_z if set(trait_z).issubset(df.columns) else [c for c in trait_raw if c in df.columns]
print("Participant trait columns used for similarity:", participant_trait_cols)

if id_col and "AdCond" in df.columns and participant_trait_cols:
    dup = (
        df.groupby(id_col)["AdCond"].nunique().describe()
        if id_col in df.columns else None
    )
    print("\nAd conditions per respondent (summary):")
    print(dup)

    # show that participant traits are often constant within respondent
    stability = []
    for c in participant_trait_cols:
        tmp = df.groupby(id_col)[c].nunique(dropna=True)
        stability.append((c, float((tmp<=1).mean()) if len(tmp) else np.nan))
    st = pd.DataFrame(stability, columns=["trait", "share_of_respondents_with_constant_value"])
    print("\nTrait constancy within respondent (higher = more constant):")
    display(st)
else:
    print("Could not run full diagnostics (missing ResponseId/AdCond/trait columns).")


## 4) Construct Table 7 ad vectors (fixed)


In [ ]:
if "AdCond" not in df.columns:
    raise ValueError("AdCond column is required.")

available_ad_signals = [c for c in ad_signal_cols if c in df.columns]

if len(available_ad_signals) == 5:
    # Preferred path: ad vectors come from ad-signal columns, not participant Big Five
    ad_vector = (
        df.dropna(subset=["AdCond"] + available_ad_signals)
          .groupby("AdCond", dropna=True)[available_ad_signals]
          .mean()
          .rename(columns={
              "OPE_cond": "Openness",
              "CON_cond": "Conscientiousness",
              "EXT_cond": "Extraversion",
              "AGR_cond": "Agreeableness",
              "NEU_cond": "Neuroticism",
          })
          .sort_index()
    )
    vector_source = "ad_signal_columns"
else:
    # Fallback: use participant trait means by ad (can be similar if repeated-measures design)
    fallback_cols = [c for c in trait_raw if c in df.columns]
    if len(fallback_cols) < 5:
        raise ValueError("Need either all *_cond ad signal columns or all trait score columns.")
    ad_vector = (
        df.dropna(subset=["AdCond"] + fallback_cols)
          .groupby("AdCond", dropna=True)[fallback_cols]
          .mean()
          .rename(columns={
              "OPEN_Score": "Openness",
              "CONSC_Score": "Conscientiousness",
              "EXTRA_Score": "Extraversion",
              "AGREE_Score": "Agreeableness",
              "NEURO_Score": "Neuroticism",
          })
          .sort_index()
    )
    vector_source = "fallback_trait_means"

print("Ad vector source:", vector_source)
table7 = ad_vector.round(3)
print("Table 7")
display(table7)


## 5) Participant-to-ad similarity (Euclidean + Cosine)


In [ ]:
# Participant vector uses z-scores when available
if set(trait_z).issubset(df.columns):
    p_cols = trait_z
    p_col_rename = {
        "ZOPEN_Score": "Openness",
        "ZCONSC_Score": "Conscientiousness",
        "ZEXTRA_Score": "Extraversion",
        "ZAGREE_Score": "Agreeableness",
        "ZNEURO_Score": "Neuroticism",
    }
elif set(trait_raw).issubset(df.columns):
    p_cols = trait_raw
    p_col_rename = {
        "OPEN_Score": "Openness",
        "CONSC_Score": "Conscientiousness",
        "EXTRA_Score": "Extraversion",
        "AGREE_Score": "Agreeableness",
        "NEURO_Score": "Neuroticism",
    }
else:
    raise ValueError("Participant trait columns not found.")

analysis_df = df.dropna(subset=["AdCond"] + p_cols).copy()
for c in p_cols:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

ad_vectors_np = ad_vector.to_dict("index")
vector_order = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))

def cosine(a, b):
    a = np.array(a).reshape(1, -1)
    b = np.array(b).reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])

def row_similarity(row):
    ad = row["AdCond"]
    if ad not in ad_vectors_np:
        return pd.Series({"euclidean": np.nan, "cosine": np.nan})
    person = [row[c] for c in p_cols]
    person_named = {p_col_rename[c]: v for c, v in zip(p_cols, person)}
    p_vec = [person_named[k] for k in vector_order]
    a_vec = [ad_vectors_np[ad][k] for k in vector_order]
    return pd.Series({"euclidean": euclidean(p_vec, a_vec), "cosine": cosine(p_vec, a_vec)})

analysis_df[["euclidean", "cosine"]] = analysis_df.apply(row_similarity, axis=1)
analysis_df[["AdCond", "euclidean", "cosine"]].head()


## 6) Table 8 (means + click correlation)


In [ ]:
def safe_corr(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 3:
        return np.nan
    return tmp["x"].corr(tmp["y"])

table8 = analysis_df.groupby("AdCond").agg(
    Euclidean_Distance_Mean=("euclidean", "mean"),
    Cosine_Similarity_Mean=("cosine", "mean"),
    Corr_Cosine_Click=("cosine", lambda s: safe_corr(s, analysis_df.loc[s.index, "CLICK_MEAN"]) if "CLICK_MEAN" in analysis_df.columns else np.nan),
    Corr_Euclidean_Click=("euclidean", lambda s: safe_corr(s, analysis_df.loc[s.index, "CLICK_MEAN"]) if "CLICK_MEAN" in analysis_df.columns else np.nan),
).round(3)

print("Table 8")
display(table8)


## 7) Table 9 cross-group validation


In [ ]:
temp = analysis_df.copy()

if "connectId" in temp.columns:
    cid_num = pd.to_numeric(temp["connectId"], errors="coerce")
    if cid_num.notna().sum() > 0:
        cutoff = cid_num.median()
        temp["grp"] = np.where(cid_num <= cutoff, "Group 1", "Group 2")
    else:
        temp["grp"] = np.where(temp["connectId"].astype(str).str.len() % 2 == 0, "Group 1", "Group 2")
elif "ResponseId" in temp.columns:
    temp["grp"] = np.where(temp["ResponseId"].astype(str).str.len() % 2 == 0, "Group 1", "Group 2")
else:
    rng = np.random.default_rng(42)
    temp["grp"] = np.where(rng.random(len(temp)) < 0.5, "Group 1", "Group 2")

pivot = temp.pivot_table(index="AdCond", columns="grp", values="cosine", aggfunc="mean")

if {"Group 1", "Group 2"}.issubset(pivot.columns):
    r_groups = pivot[["Group 1", "Group 2"]].corr().iloc[0,1]
else:
    r_groups = np.nan

table9 = pivot.rename(columns={"Group 1": "Cosine_Group1", "Group 2": "Cosine_Group2"}).round(3)
table9["r_between_groups"] = np.round(r_groups, 3) if pd.notna(r_groups) else np.nan

print("Table 9")
display(table9)


## 8) Regression comparison


In [ ]:
if "CLICK_MEAN" in analysis_df.columns:
    reg = analysis_df[["CLICK_MEAN", "euclidean", "cosine"]].dropna().copy()

    m1 = sm.OLS(reg["CLICK_MEAN"], sm.add_constant(reg[["euclidean"]])).fit()
    m2 = sm.OLS(reg["CLICK_MEAN"], sm.add_constant(reg[["cosine"]])).fit()

    print("Model A: CLICK_MEAN ~ euclidean")
    print(m1.summary())
    print("\nModel B: CLICK_MEAN ~ cosine")
    print(m2.summary())
else:
    print("CLICK_MEAN not available. Ensure CLIC1..CLIC4 are present.")


## 9) Appendix figures


In [ ]:
vector_order = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

# 9.1 3D PCA projection
if len(ad_vector) >= 3:
    pca = PCA(n_components=3)
    vec_3d = pca.fit_transform(ad_vector[vector_order].values)

    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(vec_3d[:,0], vec_3d[:,1], vec_3d[:,2], s=80)
    for i, name in enumerate(ad_vector.index):
        ax.text(vec_3d[i,0], vec_3d[i,1], vec_3d[i,2], name)
    ax.set_title("3D Projection of Advertisement 5D Trait Vectors (PCA)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    plt.show()
else:
    print("Not enough ad conditions for 3D PCA plot.")

# 9.2 Radar chart
labels = ["Open", "Consc", "Extra", "Agree", "Neuro"]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar": True})
for ad, row in ad_vector.iterrows():
    vals = [row[k] for k in vector_order]
    vals = np.concatenate([vals, [vals[0]]])
    ax.plot(angles, vals, linewidth=2, label=ad)
    ax.fill(angles, vals, alpha=0.05)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
ax.set_title("Radar Chart: Multidimensional Spillover Across Ad Conditions")
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
plt.show()

# 9.3 Cosine angle visualization (example row)
example = analysis_df.dropna(subset=p_cols + ["AdCond"]).iloc[0]
ad_name = example["AdCond"]
p_named = {p_col_rename[c]: example[c] for c in p_cols}
p_vec = np.array([p_named[k] for k in vector_order], dtype=float)
a_vec = np.array([ad_vectors_np[ad_name][k] for k in vector_order], dtype=float)

cos_val = cosine_similarity(p_vec.reshape(1,-1), a_vec.reshape(1,-1))[0,0]
angle_deg = np.degrees(np.arccos(np.clip(cos_val, -1, 1)))

plt.figure(figsize=(6,4))
plt.bar(["Cosine Similarity"], [cos_val])
plt.ylim(-1, 1)
plt.title(f"Cosine Alignment for Example Case ({ad_name})\nAngle = {angle_deg:.1f}°")
plt.show()

print(f"Cosine similarity: {cos_val:.3f}")
print(f"Angle (degrees): {angle_deg:.2f}")


## 10) Export outputs


In [ ]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

table7.to_csv(f"{out_dir}/table7_ad_profile_matrix.csv")
table8.to_csv(f"{out_dir}/table8_similarity_comparison.csv")
table9.to_csv(f"{out_dir}/table9_cross_group_validation.csv")

print("Saved files:")
for f in sorted(os.listdir(out_dir)):
    print("-", os.path.join(out_dir, f))

from google.colab import files
for f in ["table7_ad_profile_matrix.csv", "table8_similarity_comparison.csv", "table9_cross_group_validation.csv"]:
    files.download(os.path.join(out_dir, f))
